## Imports

Imports

In [1]:
import tvm
import tvm.testing
from tvm.relay import testing
from tvm import relax, relay
from tvm.relax.testing import relay_translator, nn
from tvm.runtime import vm as vm_rt
from tvm.script import relax as R
import numpy as np

[08:50:56] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[08:50:56] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[08:50:56] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[08:50:56] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[08:50:56] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[08:50:56] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled w

## Config

In [74]:
# Target
# TARGET = "llvm"
TARGET = "c"
# TARGET = tvm.target.Target("c", host="c")
target = tvm.target.Target(TARGET, host=TARGET)

# Pipeline (Relax only)
PIPELINE = "default"
# PIPELINE = "micro"
default_pipeline = "default_build"
micro_pipeline = "micro2_build"

# Exec mode (Relax only)
# EXEC_MODE = "bytecode"
EXEC_MODE = "compiled"
# EXEC_MODE = "crt"
bytecode_exec_mode = "bytecode"
compiled_exec_mode = "compiled"
crt_exec_mode = "crt"

# Executor (Relay only)
# EXECUTOR = "graph"
EXECUTOR = "aot"
aot_executor = tvm.relay.backend.Executor("aot", {"interface-api": "packed", "unpacked-api": False})
graph_executor = tvm.relay.backend.Executor("graph", {"link-params": False})

# Runtime (Relay only?)
RUNTIME = "cpp"
# RUNTIME = "crt"
SYSTEM_LIB = True
cpp_runtime = tvm.relay.backend.Runtime("cpp", {"system-lib": SYSTEM_LIB})
crt_runtime = tvm.relay.backend.Runtime("crt", {"system-lib": SYSTEM_LIB})

# Pass Config (Relay only)
USMP = False
FUSE_DEPTH = 1
VECTORIZE = False
pass_config = {"tir.usmp.enable": USMP, "relay.FuseOps.max_depth": FUSE_DEPTH, "tir.disable_vectorize": not VECTORIZE}

## Define Models

### Common

Matmul

In [48]:
matmul_input_size = 64
matmul_hidden_size = 10
matmul_output_size = 4
    
matmul_dtype = "float32"
    
matmul_weights_matrix = np.random.random((matmul_hidden_size, matmul_output_size)).astype(matmul_dtype)
matmul_bias_matrix = np.random.random((matmul_output_size,)).astype(matmul_dtype)

matmul_params = {"weights": tvm.nd.array(matmul_weights_matrix), "bias": tvm.nd.array(matmul_bias_matrix)}
matmul_data = tvm.nd.array(np.random.rand(matmul_input_size, matmul_hidden_size).astype(matmul_dtype))

Conv2d

In [49]:
conv2d_input_n = 1
conv2d_input_c = 16
conv2d_input_h = 64
conv2d_input_w = 64
conv2d_kernel_h = 4
conv2d_kernel_w = 4
conv2d_kernel_ci = 16
conv2d_kernel_co = 16
conv2d_output_n = 1
conv2d_output_c = 16
conv2d_output_h = 61
conv2d_output_w = 61
    
conv2d_dtype = "float32"
    
conv2d_weights_matrix = np.random.random((conv2d_kernel_h, conv2d_kernel_w, conv2d_kernel_ci, conv2d_kernel_co)).astype(conv2d_dtype)
conv2d_bias_matrix = np.random.random((conv2d_output_w,)).astype(conv2d_dtype)

### Relax

Define Matmul model (Relax)

In [50]:
def relax_dense():
    builder = relax.BlockBuilder()
    
    with builder.function("main"):
        input = relax.Var("x", R.Tensor((matmul_input_size, matmul_hidden_size), matmul_dtype))
        weights = relax.Constant(tvm.nd.array(matmul_weights_matrix))
        bias = relax.Constant(tvm.nd.array(matmul_bias_matrix))
        output_matmul = relax.op.matmul(input, weights)
        output_bias = relax.op.add(output_matmul, bias)
        builder.emit_func_output(output_bias, params=[input])
    return builder.get(), matmul_data, matmul_params

Define Conv2d model (Relax)

In [51]:
def relax_conv2d():
    builder = relax.BlockBuilder()
    
    with builder.function("main"):
        input = relax.Var("x", R.Tensor((input_n, input_c, input_h, input_w), conv2d_dtype))
        weights = relax.Constant(tvm.nd.array(weights_matrix))
        bias = relax.Constant(tvm.nd.array(bias_matrix))
        output_conv2d = relax.op.nn.conv2d(input, weights, data_layout="NCHW", kernel_layout="HWIO")
        output_bias = relax.op.add(output_conv2d, bias)
        builder.emit_func_output(output_bias, params=[input])
    
    return builder.get(), conv2d_data, conv2d_params

### Relay

Define Matmul model (Relay)

In [52]:
def relay_dense():

    x = relay.var("x", shape=(matmul_input_size, matmul_hidden_size), dtype=matmul_dtype)
    weight = relay.const(tvm.nd.array(matmul_weights_matrix))
    bias = relay.const(tvm.nd.array(matmul_bias_matrix))
    output_matmul = relay.nn.matmul(x, weight)
    output_bias = relay.op.add(output_matmul, bias)
    func = relay.Function(relay.analysis.free_vars(output_bias), output_bias)
    return tvm.IRModule.from_expr(func), matmul_data, matmul_params

Define Conv2D model (Relay)

In [53]:
_ = """
def relay_conv2d():
    dtype = "float32"  #  TODO: int32

    weights_matrix = np.random.random((hidden_size, output_size)).astype(dtype)
    bias_matrix = np.random.random((output_size,)).astype(dtype)

    x = relay.var("x", shape=(input_size, hidden_size), dtype=dtype)
    weight = relay.const(tvm.nd.array(weights_matrix))
    bias = relay.const(tvm.nd.array(bias_matrix))
    output_matmul = relay.nn.matmul(x, weight)
    output_bias = relay.op.add(output_matmul, bias)
    func = relay.Function(relay.analysis.free_vars(output_bias), output_bias)
    return tvm.IRModule.from_expr(func)
    return func, conv2d_data, conv2d_params
"""

## Show models

Pick used models

In [54]:
# -- Relax --
relax_mod, relax_data, relax_params = relax_dense()
# relax_mod = relax_conv2d()

# -- Relay --
relay_mod, relay_data, relay_params = relay_dense()
# relay_mod = relay_conv2d()

Show Relax module

In [55]:
relax_mod.show()

Show Relay module

In [56]:
relay_mod.show()

## Instruments

Define Pass Instrument to look at intermediate IRs during build

In [57]:
@tvm.instrument.pass_instrument
class MyInstrument:

    def __init__(self):
        self.skip_pass_name = []
        self.output = []
        self.output_after = []
        self.idx = 0

    def run_before_pass(self, mod, pass_info):
        self.idx += 1
        handle = mod.handle
        g = dict(mod.global_var_map_)
        g_ = list(g.keys())
        if len(g_) == 0:
            return
        # print(self.idx * "  " + ">", self.idx, pass_info.name, g_, len(self.output))
        # print(dir(mod))
        
        # tmp = (mod.astext(show_meta_data=True), str(pass_info))
        # tmp = (str(mod), str(pass_info))
        # tmp = (mod.script(show_meta=True), str(pass_info))
        tmp = (mod, pass_info)
        self.output.append(tmp)


    def run_after_pass(self, mod, pass_info):
        self.idx -= 1
        handle = mod.handle
        g = dict(mod.global_var_map_)
        g_ = list(g.keys())
        if len(g_) == 0:
            return
        # print((self.idx + 1) * "  " + "<", self.idx + 1, pass_info.name, g_, len(self.output_after))
        # print(dir(mod))
        # tmp = (mod.astext(show_meta_data=True), str(pass_info))
        # tmp = (str(mod), str(pass_info))
        tmp = (mod, pass_info)
        self.output_after.append(tmp)
        pass

## Build

### Relax

Relax build (VM Bytecode)

In [58]:
if PIPELINE == "default" and EXEC_MODE == "bytecode":
    relax_instrument_vm_ex_bytecode = MyInstrument()
    with tvm.transform.PassContext(instruments=[relax_instrument_vm_ex_bytecode]):
        vm_ex_bytecode = relax.build(relax_mod, target=target, pipeline=default_pipeline, exec_mode=bytecode_exec_mode)
else:
    relax_instrument_vm_ex_bytecode = None
    vm_ex_bytecode = None

Relax build (VM Compiled)

In [59]:
if PIPELINE == "default" and EXEC_MODE == "compiled":
    relax_instrument_vm_ex_compiled = MyInstrument()
    with tvm.transform.PassContext(instruments=[relax_instrument_vm_ex_compiled]):
        vm_ex_compiled = relax.build(relax_mod, target=target, pipeline=default_pipeline, exec_mode=compiled_exec_mode)
else:
    relax_instrument_vm_ex_compiled = None
    vm_ex_compiled = None

Relax build (AOT CRT)

In [60]:
if PIPELINE == "micro" and EXEC_MODE == "crt":
    relax_instrument_crt_ex_aot = MyInstrument()
    with tvm.transform.PassContext(instruments=[relax_instrument_crt_ex_aot]):
        crt_ex_aot = relax.build(relax_mod, target=target, pipeline=micro_pipeline, exec_mode=crt_exec_mode, system_lib=True)
else:
    relax_instrument_crt_ex_aot = None
    crt_ex_aot = None

### Relay

Relay build (Default C++)

In [61]:
if RUNTIME == "cpp" and EXECUTOR == "graph":
    relay_instrument_cpp_lib_default = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_cpp_lib_default], config=pass_config):
        cpp_lib_default = relay.build(relay_mod, target=target, runtime=cpp_runtime, executor=graph_executor)
else:
    relay_instrument_cpp_lib_default = None
    cpp_lib_default = None

Relay build (AoT C++)

In [62]:
if RUNTIME == "cpp" and EXECUTOR == "aot":
    relay_instrument_cpp_lib_aot = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_cpp_lib_aot], config=pass_config):
        cpp_lib_aot = relay.build(relay_mod, target=target, runtime=cpp_runtime, executor=aot_executor)
else:
    relay_instrument_cpp_lib_aot = None
    cpp_lib_aot = None

Matmul is not optimized for x86. Recommend to use cblas/mkl/dnnl for better performance.


Relay build (Graph CRT, no USMP, Packed API)

In [63]:
if RUNTIME == "crt" and EXECUTOR == "graph":
    relay_instrument_crt_lib_graph = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_crt_lib_graph], config=pass_config):
        crt_lib_graph = relay.build(relay_mod, target=target, runtime=crt_runtime, executor=graph_executor)
else:
    relay_instrument_crt_lib_graph = None
    crt_lib_graph = None

Relay build (AOT CRT, no USMP, Packed API)

In [64]:
if RUNTIME == "crt" and EXECUTOR == "aot":
    relay_instrument_crt_lib_aot = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_crt_lib_aot], config=config):
        crt_lib_aot = relay.build(relay_mod, target=target, runtime=crt_runtime, executor=aot_executor)
else:
    relay_instrument_crt_lib_aot = None
    crt_lib_aot = None

Pick compiled module

In [65]:
# -- Relax --
if PIPELINE == "default" and EXEC_MODE == "bytecode":
    ex, relax_instrument = vm_ex_bytecode, relax_instrument_vm_ex_bytecode
elif PIPELINE == "default" and EXEC_MODE == "compiled":
    ex, relax_instrument = vm_ex_compiled, relax_instrument_vm_ex_compiled
elif PIPELINE == "micro" and EXEC_MODE == "crt":
    ex, relax_instrument = crt_lib_aot, relax_instrument_crt_ex_aot
else:
    assert False

# -- Relay --
if RUNTIME == "cpp" and EXECUTOR == "graph":
    lib, relay_instrument = cpp_lib_default, relay_instrument_cpp_lib_default
elif RUNTIME == "cpp" and EXECUTOR == "aot":
    lib, relay_instrument = cpp_lib_aot, relay_instrument_cpp_lib_aot
elif RUNTIME == "crt" and EXECUTOR == "graph":
    lib, relay_instrument = crt_lib_, relay_instrument_crt_lib_graph
elif RUNTIME == "crt" and EXECUTOR == "aot":
    lib, relay_instrument = crt_lib_aot, relay_instrument_crt_lib_aot
else:
    assert False

## Investigate

Look at generated C code (Relax)

In [68]:
if TARGET == "c":
    print(ex.mod._collect_dso_modules()[0].get_source())

// tvm target: c -keys=cpu 
#define TVM_EXPORTS
#include "tvm/runtime/c_runtime_api.h"
#include "tvm/runtime/c_backend_api.h"
#include <math.h>
#include <stdbool.h>
static void* vm_builtin_check_tensor_info_packed = NULL;
static void* vm_builtin_match_shape_packed = NULL;
static void* vm_builtin_alloc_storage_packed = NULL;
static void* vm_builtin_alloc_tensor_packed = NULL;
static void* vm_builtin_null_value_packed = NULL;
static void* vm_builtin_copy_packed = NULL;
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t __vmtir__main(void* args, int32_t* arg_type_ids, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t add(void* args, int32_t* arg_type_ids, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t matmul(void* args, int32_t* arg_type_ids, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, vo

Look at generated C code (Relay)

In [75]:
if TARGET == "c":
    print(lib.lib._collect_dso_modules()[0].get_source())

// tvm target: c -keys=cpu 
#define TVM_EXPORTS
#include "tvm/runtime/c_runtime_api.h"
#include "tvm/runtime/c_backend_api.h"
#include <math.h>
#include <stdbool.h>

#ifdef __cplusplus
extern "C" {
#endif
static const float __attribute__((section(".rodata.tvm"), aligned(16))) fused_constant_1[4] = {
    0x1.23ebdap-2, 0x1.95e0b2p-1, 0x1.b14894p-1, 0x1.b27158p-3
};
#ifdef __cplusplus
}  // extern "C"
#endif

#ifdef __cplusplus
extern "C" {
#endif
static const float __attribute__((section(".rodata.tvm"), aligned(16))) fused_constant[40] = {
    0x1.fc4c28p-5, 0x1.6e005p-1, 0x1.c84accp-2, 0x1.38bafap-2, 0x1.6777e6p-1, 0x1.31b03ep-1, 0x1.afa2cap-1, 0x1.044252p-1, 
    0x1.da0fb6p-2, 0x1.afc17cp-1, 0x1.f6eccep-1, 0x1.82e28p-4, 0x1.a4c0c4p-1, 0x1.541388p-1, 0x1.35c486p-1, 0x1.7e3908p-2, 
    0x1.44d2cp-1, 0x1.923f86p-3, 0x1.cd83p-3, 0x1.118c7ep-2, 0x1.bbc82ap-1, 0x1.0623a8p-1, 0x1.848a76p-5, 0x1.13fc18p-1, 
    0x1.90679cp-1, 0x1.5ad2dep-1, 0x1.69ac9cp-1, 0x1.ed948ep-1, 0x1.a97918p-4, 0x1.53

Intermediate IRs (Relax)

In [76]:
relax_instrument.output[-1][0].show()

Intermediate IRs (Relay)

In [77]:
relay_instrument.output[-1][0].show()

## Run

### Relax

In [84]:
if TARGET == "llvm":
    if EXEC_MODE in ["bytecode", "compiled"]:
        vm = relax.VirtualMachine(ex, tvm.cpu())
        relax_output = vm["main"](relax_data).numpy()
    elif EXEC_MODE == "crt":
        pass
    else:
        assert False
else:
    relax_output = None
    print("C target does not support execution")

C target does not support execution


Show Result

In [85]:
# relax_output

### Relay

In [86]:
if TARGET == "llvm":
    dev = tvm.device(str(TARGET), dev_id=0)
    if EXECUTOR == "graph":
        rt_mod = tvm.contrib.graph_executor.GraphModule(lib["default"](dev))
    elif EXECUTOR == "aot":
        rt_mod = tvm.runtime.executor.AotModule(lib["default"](dev))
    else:
        assert False
    rt_mod.set_input("x", relay_data)
    rt_mod.run()
    relay_output = rt_mod.get_output(0).numpy()
else:
    relay_output = None
    print("C target does not support execution")

C target does not support execution


Show Result

In [87]:
# relay_output

## Compare

In [89]:
assert not(relax_output is None or relay_output is None)
tvm.testing.assert_allclose(relax_output, relay_output, rtol=1e-4, atol=1e-4)

AssertionError: 